# P1-E04 CAUEEG Event Timing and Non-Photic Eyes-Closed Interval Eligibility Audit

**Paper 1, Experiment 04 consumes the P1-E03 recording-level provenance lock**

## Summary

This notebook begins only from `11_LOCKED_RECORDING_LEVEL_PROVENANCE_MANIFEST.csv` (1,044 recordings). It authenticates that manifest, parses every event JSON without diagnostic labels, treats CAUEEG event positions as sample indices, converts each marker using its recording-specific sampling rate, audits temporal reliability, constructs eyes-closed state intervals, subtracts complete photic-stimulation and artifact windows, and locks a recording-level preprocessing-eligibility manifest.

Official train, validation, and test assignments remain unchanged. Diagnosis is joined only after interval eligibility has been frozen. Run all cells and continue only when the final status is **P1-E04 PASSED — EVENT-DEFINED RECORDING INTERVAL MANIFEST LOCKED**.


## Context & Methods

### Key assumptions and locked rules

1. P1-E03's 1,044-recording provenance manifest is the only permitted input.
2. `recording_id`/`serial` identify recordings, not independently verified patients.
3. CAUEEG event positions are sample indices. For recording (r), marker (u) is converted by (t=u/f_{s,r}).
4. Conversion is verified against EDF duration; no label-dependent or per-class unit selection is permitted.
5. Timing reliability requires at least 90% of parsed markers within the EDF duration (±2 s), no non-finite timestamps, and chronological file order.
6. Eyes-closed state begins at an Eyes Closed marker and ends at Eyes Open or Paused. Repeated state markers do not create overlapping intervals.
7. Photic On–Off pairs define full stimulation windows. Any recording with an unmatched Photic On or Photic Off marker is classified as a malformed photic sequence and excluded from eligibility.
8. Photic windows are expanded by 2 s before and 10 s after. Artifact/movement markers are expanded by 1 s before and 2 s after.
9. After subtraction, retained pieces shorter than 10 s are discarded.
10. Thresholds are fixed before diagnosis is joined. The test split must not influence later preprocessing or model choices.


## Data

### 1. Setup, paths and output contract


In [ ]:
from pathlib import Path
import json, re, sys, platform, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

DATA_ROOT = Path(r'D:\EEG_AD_Research\data\caueeg-dataset\caueeg-dataset')
PROJECT_ROOT = Path(r'D:\EEG_AD_Research')
PAPER1_ROOT = PROJECT_ROOT/'paper1_experiments'
E03_ROOT = PAPER1_ROOT/'paper1_outputs'/'p1_e03_recording_linkage_lock'
INPUT_MANIFEST = E03_ROOT/'tables'/'11_LOCKED_RECORDING_LEVEL_PROVENANCE_MANIFEST.csv'
E03_RUN_RECORD = E03_ROOT/'run_record.json'
OUTPUT_ROOT = PAPER1_ROOT/'paper1_outputs'/'p1_e04_event_interval_audit'
TABLE_DIR, FIGURE_DIR = OUTPUT_ROOT/'tables', OUTPUT_ROOT/'figures'
for directory in (OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

CLASS_ORDER = ['Normal','MCI','AD']
SPLIT_ORDER = ['train','validation','test']
EXPECTED_E03_COUNTS = pd.DataFrame(
    {'Normal':[366,34,35], 'MCI':[334,28,33], 'AD':[189,10,15]},
    index=SPLIT_ORDER,
)
CLASS_COLORS = {'Normal':'#0072B2','MCI':'#E69F00','AD':'#D55E00'}
MIN_INTERVAL_SECONDS = 10.0
TIMING_TOLERANCE_SECONDS = 2.0
PHOTIC_PAD_BEFORE_SECONDS = 2.0
PHOTIC_PAD_AFTER_SECONDS = 10.0
ARTIFACT_PAD_BEFORE_SECONDS = 1.0
ARTIFACT_PAD_AFTER_SECONDS = 2.0
MIN_RECORDING_IN_BOUNDS = 0.90
MIN_GLOBAL_IN_BOUNDS = 0.95
MIN_MEDIAN_RECORDING_IN_BOUNDS = 0.99
MAX_FRACTION_RECORDINGS_UNRELIABLE = 0.05

mpl.rcParams.update({'font.family':'Arial','font.size':8,'axes.titlesize':9,
    'axes.labelsize':8,'legend.fontsize':7,'pdf.fonttype':42,'ps.fonttype':42,
    'figure.dpi':120,'axes.spines.top':False,'axes.spines.right':False})
sns.set_theme(style='ticks', context='paper')

def sha256_file(path, block_size=4*1024*1024):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(block_size),b''):
            digest.update(block)
    return digest.hexdigest()

def save_table(frame,name,index=False):
    frame.to_csv(TABLE_DIR/f'{name}.csv',index=index)
    try:
        frame.to_excel(TABLE_DIR/f'{name}.xlsx',index=index)
    except Exception as exc:
        warnings.warn(f'Excel export skipped ({exc}); CSV was saved.')

def save_figure(fig,name,width_mm=180,height_mm=90):
    fig.set_size_inches(width_mm/25.4,height_mm/25.4)
    fig.tight_layout()
    for ext in ('png','pdf','svg'):
        fig.savefig(FIGURE_DIR/f'{name}.{ext}',dpi=600 if ext=='png' else None,
                    bbox_inches='tight',facecolor='white')

def norm_text(value):
    return re.sub(r'\s+',' ',str(value).strip().lower())

required_inputs={'P1-E03 manifest':INPUT_MANIFEST,'P1-E03 run record':E03_RUN_RECORD}
missing=[f'{name}: {path}' for name,path in required_inputs.items() if not path.is_file()]
if missing:
    raise FileNotFoundError('Required P1-E03 outputs are missing:\n'+'\n'.join(missing))
print('Input manifest:',INPUT_MANIFEST)
print('Output root:',OUTPUT_ROOT)


### 2. Authenticate the completed P1-E03 recording manifest

The source must reproduce the final 1,044-record cohort and its immutable class-by-split counts. Its SHA-256 is compared with the P1-E03 run record.


In [ ]:
with E03_RUN_RECORD.open('r',encoding='utf-8') as handle:
    e03_run=json.load(handle)
expected_hash=e03_run.get('manifest_sha256','')
observed_hash=sha256_file(INPUT_MANIFEST)
if not expected_hash or observed_hash!=expected_hash:
    raise AssertionError('STOP — P1-E03 manifest SHA-256 is absent or mismatched.')

locked=pd.read_csv(INPUT_MANIFEST,dtype={'recording_id':str,'serial':str,'identity_group':str})
required={'recording_id','serial','age','official_split','experimental_label',
          'experimental_label_code','analysis_unit','identity_group',
          'participant_identity_verified','split_source','event_path','edf_path',
          'duration_seconds','sfreq_Hz','n_samples','manifest_row_sha256'}
missing_columns=required-set(locked.columns)
if missing_columns:
    raise ValueError(f'P1-E03 manifest lacks required columns: {sorted(missing_columns)}')

for column in ('recording_id','serial','identity_group'):
    locked[column]=locked[column].astype(str).str.strip().str.zfill(5)
locked['official_split']=locked.official_split.astype(str).str.strip().str.lower()
locked['experimental_label']=locked.experimental_label.astype(str).str.strip()
for column in ('duration_seconds','sfreq_Hz','n_samples'):
    locked[column]=pd.to_numeric(locked[column],errors='coerce')

observed_counts=(locked.groupby(['official_split','experimental_label']).size()
    .unstack(fill_value=0).reindex(index=SPLIT_ORDER,columns=CLASS_ORDER,fill_value=0).astype(int))
duration_error=(locked.duration_seconds-locked.n_samples/locked.sfreq_Hz).abs()
input_checks=pd.DataFrame([
 {'check':'P1-E03 manifest SHA-256 authenticated','pass':observed_hash==expected_hash,'detail':observed_hash},
 {'check':'exactly 1,044 recordings loaded','pass':len(locked)==1044,'detail':len(locked)},
 {'check':'recording IDs and serials unique','pass':not locked.recording_id.duplicated().any() and not locked.serial.duplicated().any(),'detail':{'recording_ids':locked.recording_id.nunique(),'serials':locked.serial.nunique()}},
 {'check':'recording ID equals released serial','pass':locked.recording_id.eq(locked.serial).all(),'detail':int((locked.recording_id!=locked.serial).sum())},
 {'check':'class-by-split counts reproduce P1-E03','pass':observed_counts.equals(EXPECTED_E03_COUNTS),'detail':observed_counts.to_dict()},
 {'check':'analysis unit remains recording','pass':locked.analysis_unit.astype(str).str.lower().eq('recording').all(),'detail':locked.analysis_unit.value_counts().to_dict()},
 {'check':'participant identity remains unverified','pass':~locked.participant_identity_verified.astype(str).str.lower().isin(['true','1']).any(),'detail':locked.participant_identity_verified.value_counts(dropna=False).to_dict()},
 {'check':'EDF timing fields complete and positive','pass':locked[['duration_seconds','sfreq_Hz','n_samples']].notna().all().all() and (locked[['duration_seconds','sfreq_Hz','n_samples']]>0).all().all(),'detail':locked[['duration_seconds','sfreq_Hz','n_samples']].isna().sum().to_dict()},
 {'check':'EDF duration equals samples divided by sampling rate','pass':duration_error.max()<=1/locked.sfreq_Hz.min()+1e-9,'detail':float(duration_error.max())},
 {'check':'all event JSON paths exist','pass':locked.event_path.map(lambda p:Path(p).is_file()).all(),'detail':int((~locked.event_path.map(lambda p:Path(p).is_file())).sum())},
])
save_table(input_checks,'01_p1_e03_input_authentication_checks')
display(observed_counts); display(input_checks)
if not input_checks['pass'].all():
    raise AssertionError('STOP — P1-E03 INPUT AUTHENTICATION FAILED')
print('P1-E03 INPUT AUTHENTICATED')


## Results

### 3. Parse all event JSON files without diagnostic labels

Only recording identity, path and timing metadata are exposed to the parser and interval constructor.


In [ ]:
TIME_KEYS=('time','timestamp','latency','onset','position','sample','start')
LABEL_KEYS=('label','event','description','text','name','type','value')
DURATION_KEYS=('duration','length','span')

def first_scalar(mapping,keys):
    for key in keys:
        if key in mapping and not isinstance(mapping[key],(dict,list,tuple)):
            return mapping[key]
    return None

def collect_events(obj,out):
    if isinstance(obj,dict):
        t=first_scalar(obj,TIME_KEYS); label=first_scalar(obj,LABEL_KEYS)
        duration=first_scalar(obj,DURATION_KEYS)
        if t is not None and label is not None:
            try:
                out.append({'raw_sample':float(t),'label':str(label),
                            'raw_duration_samples':float(duration) if duration is not None else np.nan})
            except (TypeError,ValueError):
                pass
        for value in obj.values():
            collect_events(value,out)
    elif isinstance(obj,(list,tuple)):
        if len(obj)>=2 and np.isscalar(obj[0]) and isinstance(obj[1],str):
            try:
                out.append({'raw_sample':float(obj[0]),'label':obj[1],
                            'raw_duration_samples':float(obj[2]) if len(obj)>2 and np.isscalar(obj[2]) else np.nan})
            except (TypeError,ValueError):
                pass
        else:
            for value in obj:
                collect_events(value,out)

event_rows=[]; parse_rows=[]
blind_columns=['recording_id','serial','event_path','duration_seconds','sfreq_Hz','n_samples']
for row in locked[blind_columns].itertuples(index=False):
    try:
        payload=json.loads(Path(row.event_path).read_text(encoding='utf-8-sig'))
        found=[]; collect_events(payload,found)
        for event_number,event in enumerate(found):
            event_rows.append({
                'recording_id':row.recording_id,'serial':row.serial,
                'event_number':event_number,'raw_sample':event['raw_sample'],
                'raw_duration_samples':event['raw_duration_samples'],'label':event['label'],
                'label_normalized':norm_text(event['label']),
                'recording_duration_seconds':row.duration_seconds,
                'sfreq_Hz':row.sfreq_Hz,'n_samples':row.n_samples,
            })
        parse_rows.append({'recording_id':row.recording_id,'serial':row.serial,
                           'readable':True,'parsed_events':len(found),'error':''})
    except Exception as exc:
        parse_rows.append({'recording_id':row.recording_id,'serial':row.serial,
                           'readable':False,'parsed_events':0,'error':repr(exc)})

events=pd.DataFrame(event_rows); parse_audit=pd.DataFrame(parse_rows)
if events.empty:
    raise ValueError('No events were parsed.')
label_catalog=(events.groupby('label_normalized').agg(
    events=('label','size'),recordings=('serial','nunique'),example=('label','first'))
    .sort_values(['recordings','events'],ascending=False).reset_index())
save_table(parse_audit,'02_event_file_parse_audit')
save_table(label_catalog,'03_event_label_catalog')
display(parse_audit.parsed_events.describe().to_frame())
display(label_catalog.head(35))


### 4. Convert sample indices to seconds and verify timing

For each recording (r):

\[
t_{e,r}=\frac{u_{e,r}}{f_{s,r}}.
\]

This cell validates the documented/sample-index interpretation using EDF duration. It does not search for a label-dependent conversion factor.


In [ ]:
events['time_seconds']=events.raw_sample/events.sfreq_Hz
events['event_duration_seconds']=events.raw_duration_samples/events.sfreq_Hz
events['finite_time']=np.isfinite(events.time_seconds)
events['in_bounds']=(events.finite_time &
    (events.time_seconds>=-TIMING_TOLERANCE_SECONDS) &
    (events.time_seconds<=events.recording_duration_seconds+TIMING_TOLERANCE_SECONDS))

def event_kind(label):
    s=norm_text(label)
    if re.search(r'eyes?\s*closed|eye\s*close|\bec\b',s): return 'eyes_closed'
    if re.search(r'eyes?\s*open|eye\s*open|\beo\b',s): return 'eyes_open'
    if re.search(r'photic\s*on|photo\s*on|flash\s*on|stimulation\s*on',s): return 'photic_on'
    if re.search(r'photic\s*off|photo\s*off|flash\s*off|stimulation\s*off',s): return 'photic_off'
    if s=='paused' or re.search(r'pause recording|recording paused',s): return 'paused'
    if re.search(r'recording resumed|resume recording',s): return 'resumed'
    if re.search(r'artifact|artefact|move|movement|talk|cough|chew|swallow|blink|electrode|recheck',s): return 'artifact'
    return 'other'

events['event_kind']=events.label_normalized.map(event_kind)
events['raw_order_inversion']=events.groupby('recording_id').raw_sample.diff().lt(0)
temporal_audit=(events.groupby(['recording_id','serial']).agg(
    events=('event_number','size'),finite_fraction=('finite_time','mean'),
    within_fraction=('in_bounds','mean'),order_inversions=('raw_order_inversion','sum'),
    eyes_closed_markers=('event_kind',lambda x:(x=='eyes_closed').sum()),
    eyes_open_markers=('event_kind',lambda x:(x=='eyes_open').sum()),
    photic_on_markers=('event_kind',lambda x:(x=='photic_on').sum()),
    photic_off_markers=('event_kind',lambda x:(x=='photic_off').sum()),
    artifact_markers=('event_kind',lambda x:(x=='artifact').sum()),
    max_event_seconds=('time_seconds','max'),
    recording_duration_seconds=('recording_duration_seconds','first')).reset_index())
temporal_audit['max_event_to_duration_ratio']=temporal_audit.max_event_seconds/temporal_audit.recording_duration_seconds
temporal_audit['timing_reliable']=(
    temporal_audit.finite_fraction.eq(1.0) &
    temporal_audit.within_fraction.ge(MIN_RECORDING_IN_BOUNDS) &
    temporal_audit.order_inversions.eq(0)
)

global_in_bounds=float(events.in_bounds.mean())
median_in_bounds=float(temporal_audit.within_fraction.median())
fraction_unreliable=float((~temporal_audit.timing_reliable).mean())
out_of_bounds=events.loc[~events.in_bounds].copy()
kind_counts=events.event_kind.value_counts().rename_axis('event_kind').reset_index(name='markers')
conversion_audit=pd.DataFrame([{
    'timestamp_interpretation':'sample_index','conversion':'raw_sample / recording sfreq_Hz',
    'sampling_rates_Hz':' | '.join(map(str,sorted(events.sfreq_Hz.unique()))),
    'global_in_bounds_fraction':global_in_bounds,
    'median_recording_in_bounds_fraction':median_in_bounds,
    'median_max_event_to_duration_ratio':float(temporal_audit.max_event_to_duration_ratio.median()),
    'p05_max_event_to_duration_ratio':float(temporal_audit.max_event_to_duration_ratio.quantile(.05)),
    'p95_max_event_to_duration_ratio':float(temporal_audit.max_event_to_duration_ratio.quantile(.95)),
}])
save_table(conversion_audit,'04_sample_index_conversion_verification')
save_table(events,'05_normalized_event_marker_audit')
save_table(temporal_audit,'06_recording_event_timing_reliability')
save_table(out_of_bounds,'07_out_of_bounds_event_markers')
save_table(kind_counts,'08_event_kind_counts')
display(conversion_audit); display(kind_counts)
print(f'Global in-bounds: {global_in_bounds:.4%} | median recording: {median_in_bounds:.4%} | unreliable recordings: {fraction_unreliable:.4%}')


### 5. Diagnosis-blinded eyes-closed state and exclusion-window construction

Eyes-closed intervals and exclusion windows are generated using no diagnosis or split labels. Full Photic On–Off blocks are removed, not merely padded marker points.


In [ ]:
def merge_intervals(intervals):
    cleaned=sorted((float(a),float(b)) for a,b in intervals
                   if np.isfinite(a) and np.isfinite(b) and b>a)
    merged=[]
    for start,stop in cleaned:
        if not merged or start>merged[-1][1]:
            merged.append([start,stop])
        else:
            merged[-1][1]=max(merged[-1][1],stop)
    return [(a,b) for a,b in merged]

def subtract_windows(base,windows):
    pieces=[base]
    for window_start,window_stop in merge_intervals(windows):
        new=[]
        for start,stop in pieces:
            if window_stop<=start or window_start>=stop:
                new.append((start,stop))
            else:
                if window_start>start: new.append((start,min(window_start,stop)))
                if window_stop<stop: new.append((max(window_stop,start),stop))
        pieces=new
    return pieces

def construct_recording_intervals(record_events,duration_seconds):
    ev=record_events.loc[record_events.in_bounds].copy()
    ev=ev.sort_values(['time_seconds','event_number'])
    base=[]; eyes_closed_start=None
    photic_windows=[]; photic_start=None; unmatched_photic_off=0
    artifact_windows=[]

    for row in ev.itertuples(index=False):
        t=float(np.clip(row.time_seconds,0,duration_seconds))
        if row.event_kind=='eyes_closed':
            if eyes_closed_start is None: eyes_closed_start=t
        elif row.event_kind in ('eyes_open','paused'):
            if eyes_closed_start is not None and t>eyes_closed_start:
                base.append((eyes_closed_start,t))
            eyes_closed_start=None
        if row.event_kind=='photic_on':
            if photic_start is None: photic_start=t
        elif row.event_kind=='photic_off':
            if photic_start is not None:
                photic_windows.append((max(0,photic_start-PHOTIC_PAD_BEFORE_SECONDS),
                                       min(duration_seconds,t+PHOTIC_PAD_AFTER_SECONDS)))
                photic_start=None
            else:
                unmatched_photic_off+=1
                photic_windows.append((max(0,t-PHOTIC_PAD_BEFORE_SECONDS),
                                       min(duration_seconds,t+PHOTIC_PAD_AFTER_SECONDS)))
        elif row.event_kind=='artifact':
            artifact_windows.append((max(0,t-ARTIFACT_PAD_BEFORE_SECONDS),
                                     min(duration_seconds,t+ARTIFACT_PAD_AFTER_SECONDS)))

    if eyes_closed_start is not None and duration_seconds>eyes_closed_start:
        base.append((eyes_closed_start,duration_seconds))
    unmatched_photic_on=int(photic_start is not None)
    if photic_start is not None:
        photic_windows.append((max(0,photic_start-PHOTIC_PAD_BEFORE_SECONDS),duration_seconds))

    base=merge_intervals(base)
    exclusions=merge_intervals(photic_windows+artifact_windows)
    retained=[]
    for interval in base:
        retained.extend(subtract_windows(interval,exclusions))
    retained=merge_intervals(retained)
    retained=[(a,b) for a,b in retained if b-a>=MIN_INTERVAL_SECONDS]
    diagnostics={'base_intervals':len(base),'photic_windows':len(merge_intervals(photic_windows)),
                 'artifact_windows':len(merge_intervals(artifact_windows)),
                 'exclusion_windows':len(exclusions),'unmatched_photic_on':unmatched_photic_on,
                 'unmatched_photic_off':unmatched_photic_off}
    return base,exclusions,retained,diagnostics

timing_map=temporal_audit.set_index('recording_id').timing_reliable.to_dict()
interval_rows=[]; eligibility_rows=[]; construction_rows=[]
for row in locked[['recording_id','serial','duration_seconds']].itertuples(index=False):
    record_events=events.loc[events.recording_id.eq(row.recording_id)]
    reliable=bool(timing_map.get(row.recording_id,False))
    if reliable:
        base,exclusions,retained,diag=construct_recording_intervals(record_events,float(row.duration_seconds))
    else:
        base,exclusions,retained,diag=[],[],[],{'base_intervals':0,'photic_windows':0,
            'artifact_windows':0,'exclusion_windows':0,'unmatched_photic_on':0,'unmatched_photic_off':0}
    candidate_retained=list(retained)
    malformed_photic_sequence=bool(
        reliable and (diag['unmatched_photic_on']>0 or diag['unmatched_photic_off']>0)
    )
    # A missing boundary makes the true stimulation window unknowable. Do not
    # permit any candidate interval from that recording into the locked output.
    if malformed_photic_sequence:
        retained=[]
    for number,(start,stop) in enumerate(retained,1):
        interval_rows.append({'recording_id':row.recording_id,'serial':row.serial,
            'interval_number':number,'start_seconds':start,'stop_seconds':stop,
            'duration_seconds':stop-start,'timestamp_unit':'sample_index',
            'conversion_rule':'raw_sample / recording sfreq_Hz',
            'interval_rule_version':'P1-E04-v3-strict-photic-sequence'})
    if not reliable: reason='timing_unreliable'
    elif malformed_photic_sequence: reason='malformed_photic_sequence'
    elif not (record_events.event_kind=='eyes_closed').any(): reason='no_eyes_closed_marker'
    elif not base: reason='no_valid_eyes_closed_state_interval'
    elif not retained: reason='all_intervals_removed_or_below_10_seconds'
    else: reason='eligible'
    eligibility_rows.append({'recording_id':row.recording_id,'serial':row.serial,
        'timing_reliable':reliable,'malformed_photic_sequence':malformed_photic_sequence,
        **diag,'candidate_retained_intervals_before_photic_gate':len(candidate_retained),
        'candidate_retained_seconds_before_photic_gate':sum(b-a for a,b in candidate_retained),
        'retained_intervals':len(retained),
        'retained_seconds':sum(b-a for a,b in retained),'eligible':bool(retained),'reason':reason})
    construction_rows.append({'recording_id':row.recording_id,'serial':row.serial,
        'base_eyes_closed_seconds':sum(b-a for a,b in base),
        'excluded_window_seconds':sum(b-a for a,b in exclusions),
        'malformed_photic_sequence':malformed_photic_sequence,
        'candidate_retained_intervals_before_photic_gate':len(candidate_retained),
        'candidate_retained_seconds_before_photic_gate':sum(b-a for a,b in candidate_retained),**diag})

intervals=pd.DataFrame(interval_rows)
eligibility_blind=pd.DataFrame(eligibility_rows)
construction_audit=pd.DataFrame(construction_rows)
save_table(construction_audit,'09_diagnosis_blinded_interval_construction_audit')
save_table(intervals,'10_LOCKED_NONPHOTIC_EYES_CLOSED_INTERVALS')
save_table(eligibility_blind,'11_diagnosis_blinded_recording_eligibility')
malformed_photic_blind=eligibility_blind.loc[eligibility_blind.malformed_photic_sequence].copy()
save_table(malformed_photic_blind,'12_MALFORMED_PHOTIC_SEQUENCE_RECORDINGS_DIAGNOSIS_BLINDED')
display(eligibility_blind.reason.value_counts().rename_axis('reason').reset_index(name='recordings'))
display(pd.DataFrame([{
    'malformed_photic_recordings':len(malformed_photic_blind),
    'unmatched_photic_on_markers':int(malformed_photic_blind.unmatched_photic_on.sum()),
    'unmatched_photic_off_markers':int(malformed_photic_blind.unmatched_photic_off.sum()),
}]))


### 6. Join labels only after eligibility is frozen and preserve official splits


In [ ]:
audit_fields=['recording_id','official_split','experimental_label','experimental_label_code',
              'age','analysis_unit','identity_group','participant_identity_verified','split_source',
              'edf_path','event_path','duration_seconds','sfreq_Hz','n_samples',
              'edf_sha256','event_sha256','manifest_row_sha256']
audit_fields=[c for c in audit_fields if c in locked.columns]
eligibility=eligibility_blind.merge(locked[audit_fields],on='recording_id',how='left',validate='one_to_one')
eligible=eligibility.loc[eligibility.eligible].copy()

summary=(eligibility.groupby(['official_split','experimental_label']).agg(
    input_recordings=('recording_id','size'),eligible_recordings=('eligible','sum'),
    median_retained_minutes=('retained_seconds',lambda x:np.median(x)/60),
    p10_retained_minutes=('retained_seconds',lambda x:np.quantile(x,.10)/60),
    p90_retained_minutes=('retained_seconds',lambda x:np.quantile(x,.90)/60)).reset_index())
summary['eligibility_fraction']=summary.eligible_recordings/summary.input_recordings
reason_summary=(eligibility.groupby(['official_split','experimental_label','reason']).size()
                .rename('recordings').reset_index())

interval_totals=(intervals.groupby('recording_id').agg(
    retained_interval_count=('interval_number','size'),
    retained_seconds_verified=('duration_seconds','sum')).reset_index())
final_manifest=eligible.merge(interval_totals,on='recording_id',how='left',validate='one_to_one')
final_manifest['interval_manifest_version']='P1-E04-v3-strict-photic-sequence'
final_manifest['source_p1_e03_manifest_sha256']=observed_hash
final_manifest=final_manifest.sort_values(['official_split','experimental_label','recording_id']).reset_index(drop=True)

malformed_photic_labeled=eligibility.loc[eligibility.malformed_photic_sequence].copy()
save_table(eligibility,'13_recording_eligibility_with_labels_for_audit_only')
save_table(summary,'14_eligibility_summary_by_split_and_class')
save_table(reason_summary,'15_ineligibility_reasons_by_split_and_class')
save_table(malformed_photic_labeled,'16_malformed_photic_sequences_by_split_and_class')
save_table(final_manifest,'17_LOCKED_PREPROCESSING_ELIGIBLE_RECORDING_MANIFEST')
display(summary); display(reason_summary)


### 7. Publication figures


In [ ]:
class_summary=(eligibility.groupby('experimental_label').agg(
    input_recordings=('recording_id','size'),eligible_recordings=('eligible','sum'))
    .reindex(CLASS_ORDER).reset_index())
fig,axes=plt.subplots(1,2)
x=np.arange(len(CLASS_ORDER))
axes[0].bar(x-.18,class_summary.input_recordings,.36,label='P1-E03 input',color='#BDBDBD')
axes[0].bar(x+.18,class_summary.eligible_recordings,.36,label='P1-E04 eligible',
            color=[CLASS_COLORS[c] for c in CLASS_ORDER])
axes[0].set(xticks=x,xticklabels=CLASS_ORDER,ylabel='Recordings',title='Event-interval eligibility retention')
axes[0].legend(frameon=False)
for container in axes[0].containers: axes[0].bar_label(container,padding=2,fontsize=7)
sns.boxplot(data=eligible,x='experimental_label',y=eligible.retained_seconds/60,
            order=CLASS_ORDER,hue='experimental_label',palette=CLASS_COLORS,
            legend=False,ax=axes[1])
axes[1].set(xlabel='Diagnostic group',ylabel='Retained non-photic eyes-closed time (minutes)',
            title='Usable event-defined duration')
save_figure(fig,'figure_1_interval_eligibility_and_retained_duration',180,82); plt.show()

fig,axes=plt.subplots(1,2)
sns.histplot(temporal_audit.within_fraction,bins=25,color='#0072B2',ax=axes[0])
axes[0].axvline(MIN_RECORDING_IN_BOUNDS,color='#D55E00',ls='--',label='Reliability threshold')
axes[0].set(xlabel='Within-recording marker fraction',ylabel='Recordings',title='Sample-index timing reliability')
axes[0].legend(frameon=False)
plot_kinds=kind_counts.set_index('event_kind').reindex(
    ['eyes_closed','eyes_open','photic_on','photic_off','artifact','paused','resumed','other'],fill_value=0)
axes[1].bar(plot_kinds.index,plot_kinds.markers,color='#56B4E9')
axes[1].tick_params(axis='x',rotation=35); axes[1].set(ylabel='Markers',title='Parsed event categories')
save_figure(fig,'figure_2_timing_reliability_and_event_categories',180,82); plt.show()

flow=pd.DataFrame([
 {'stage':'P1-E03 provenance-locked recordings','recordings':len(locked)},
 {'stage':'Timing-reliable recordings','recordings':int(eligibility.timing_reliable.sum())},
 {'stage':'P1-E04 preprocessing-eligible recordings','recordings':int(eligibility.eligible.sum())}])
save_table(flow,'18_recording_interval_flow')
fig,ax=plt.subplots(); bars=ax.barh(flow.stage,flow.recordings,color=['#999999','#56B4E9','#009E73'])
ax.invert_yaxis(); ax.bar_label(bars,padding=3)
ax.set(xlabel='Recordings',title='P1-E04 event-defined interval eligibility flow')
save_figure(fig,'figure_3_recording_interval_eligibility_flow',165,75); plt.show()


### 8. Final integrity gates and reproducibility record

The next experiment must not begin unless every check passes.


In [ ]:
overlaps=outside=too_short=0
if not intervals.empty:
    for recording_id,group in intervals.sort_values(['recording_id','start_seconds']).groupby('recording_id'):
        if len(group)>1:
            overlaps+=int((group.start_seconds.to_numpy()[1:]<group.stop_seconds.to_numpy()[:-1]-1e-9).sum())
        duration=float(locked.loc[locked.recording_id.eq(recording_id),'duration_seconds'].iloc[0])
        outside+=int(((group.start_seconds<0)|(group.stop_seconds>duration+1e-9)).sum())
        too_short+=int((group.duration_seconds<MIN_INTERVAL_SECONDS-1e-9).sum())

eligible_counts=(eligible.groupby(['official_split','experimental_label']).size().unstack(fill_value=0)
    .reindex(index=SPLIT_ORDER,columns=CLASS_ORDER,fill_value=0).astype(int))
verified_total_difference=(final_manifest.retained_seconds-final_manifest.retained_seconds_verified).abs().max()
checks=pd.DataFrame([
 {'check':'P1-E03 source authenticated','pass':observed_hash==expected_hash and len(locked)==1044,'detail':len(locked)},
 {'check':'one eligibility row per locked recording','pass':len(eligibility)==len(locked) and not eligibility.recording_id.duplicated().any(),'detail':len(eligibility)},
 {'check':'all event JSON files parsed with events','pass':parse_audit.readable.all() and (parse_audit.parsed_events>0).all(),'detail':f'{int(parse_audit.readable.sum())}/{len(parse_audit)}'},
 {'check':'sample-index conversion uses recording sampling rate','pass':events.sfreq_Hz.notna().all() and (events.sfreq_Hz>0).all(),'detail':sorted(events.sfreq_Hz.unique().tolist())},
 {'check':'global timing in-bounds threshold passed','pass':global_in_bounds>=MIN_GLOBAL_IN_BOUNDS,'detail':global_in_bounds},
 {'check':'median recording timing threshold passed','pass':median_in_bounds>=MIN_MEDIAN_RECORDING_IN_BOUNDS,'detail':median_in_bounds},
 {'check':'unreliable-recording fraction acceptable','pass':fraction_unreliable<=MAX_FRACTION_RECORDINGS_UNRELIABLE,'detail':fraction_unreliable},
 {'check':'timing-unreliable recordings excluded','pass':not eligibility.loc[~eligibility.timing_reliable,'eligible'].any(),'detail':int((~eligibility.timing_reliable).sum())},
 {'check':'all malformed photic sequences documented','pass':len(malformed_photic_labeled)==int(eligibility.malformed_photic_sequence.sum()),'detail':{'recordings':len(malformed_photic_labeled),'unmatched_on':int(malformed_photic_labeled.unmatched_photic_on.sum()),'unmatched_off':int(malformed_photic_labeled.unmatched_photic_off.sum())}},
 {'check':'malformed photic sequences excluded from eligibility','pass':not eligibility.loc[eligibility.malformed_photic_sequence,'eligible'].any(),'detail':int(eligibility.malformed_photic_sequence.sum())},
 {'check':'eligible recordings have fully paired photic markers','pass':not final_manifest.malformed_photic_sequence.any() and final_manifest[['unmatched_photic_on','unmatched_photic_off']].sum().sum()==0,'detail':final_manifest[['unmatched_photic_on','unmatched_photic_off']].sum().to_dict()},
 {'check':'retained intervals are non-overlapping','pass':overlaps==0,'detail':overlaps},
 {'check':'retained intervals meet minimum duration','pass':too_short==0 and not intervals.empty,'detail':{'too_short':too_short,'minimum':float(intervals.duration_seconds.min()) if not intervals.empty else None}},
 {'check':'retained intervals lie inside EDF duration','pass':outside==0,'detail':outside},
 {'check':'interval totals reconcile to eligibility manifest','pass':verified_total_difference<=1e-8,'detail':float(verified_total_difference)},
 {'check':'official split assignments preserved','pass':set(final_manifest.official_split).issubset(set(SPLIT_ORDER)),'detail':final_manifest.official_split.value_counts().to_dict()},
 {'check':'all endpoint classes remain in every split','pass':eligible_counts.gt(0).all().all(),'detail':eligible_counts.to_dict()},
 {'check':'analysis remains recording-level','pass':final_manifest.analysis_unit.astype(str).str.lower().eq('recording').all(),'detail':final_manifest.analysis_unit.value_counts().to_dict()},
 {'check':'participant identity remains unverified','pass':~final_manifest.participant_identity_verified.astype(str).str.lower().isin(['true','1']).any(),'detail':final_manifest.participant_identity_verified.value_counts(dropna=False).to_dict()},
 {'check':'diagnosis-blinded construction preserved','pass':'experimental_label' not in eligibility_blind.columns and 'experimental_label' not in intervals.columns,'detail':'labels joined only after eligibility lock'},
])
save_table(checks,'19_final_integrity_checks')
display(checks)
status=('P1-E04 PASSED — EVENT-DEFINED RECORDING INTERVAL MANIFEST LOCKED'
        if checks['pass'].all() else 'STOP — P1-E04 CHECKS FAILED; REVIEW BEFORE CHANNEL HARMONISATION')

interval_csv=TABLE_DIR/'10_LOCKED_NONPHOTIC_EYES_CLOSED_INTERVALS.csv'
eligible_csv=TABLE_DIR/'17_LOCKED_PREPROCESSING_ELIGIBLE_RECORDING_MANIFEST.csv'
run_record={
 'experiment':'P1-E04 CAUEEG event timing and non-photic eyes-closed interval eligibility audit',
 'timestamp_utc':pd.Timestamp.utcnow().isoformat(),
 'source_manifest':str(INPUT_MANIFEST),'source_manifest_sha256':observed_hash,
 'source_recordings':len(locked),'eligible_recordings':len(final_manifest),
 'eligible_class_counts':{str(k):int(v) for k,v in final_manifest.experimental_label.value_counts().items()},
 'eligible_split_counts':{str(k):int(v) for k,v in final_manifest.official_split.value_counts().items()},
 'timestamp_interpretation':'sample_index','conversion_rule':'raw_sample / recording sfreq_Hz',
 'global_event_in_bounds_fraction':global_in_bounds,
 'median_recording_in_bounds_fraction':median_in_bounds,
 'unreliable_recording_fraction':fraction_unreliable,'intervals':len(intervals),
 'malformed_photic_sequence_recordings':len(malformed_photic_labeled),
 'unmatched_photic_on_markers':int(malformed_photic_labeled.unmatched_photic_on.sum()),
 'unmatched_photic_off_markers':int(malformed_photic_labeled.unmatched_photic_off.sum()),
 'locked_interval_csv':str(interval_csv),'locked_interval_sha256':sha256_file(interval_csv),
 'preprocessing_eligible_manifest_csv':str(eligible_csv),
 'preprocessing_eligible_manifest_sha256':sha256_file(eligible_csv),
 'identity_interpretation':{'analysis_unit':'recording','serial_to_patient_mapping_available':False,
   'held_out_protection':'official CAUEEG no-overlap split carried forward unchanged'},
 'python':sys.version,'platform':platform.platform(),'status':status}
(OUTPUT_ROOT/'run_record.json').write_text(json.dumps(run_record,indent=2),encoding='utf-8')
print(json.dumps(run_record,indent=2))
if not checks['pass'].all():
    raise AssertionError(status)


## Takeaways and next step

1. CAUEEG event positions are handled as sample indices and converted by each recording's sampling rate.
2. Eligibility is diagnosis-blinded and recording-level; no participant identity is inferred from serial numbers.
3. Full Photic On–Off blocks and artifact windows are removed from eyes-closed state intervals; recordings with any unmatched photic boundary are documented and excluded.
4. Official train, validation, and test assignments remain unchanged.
5. The only permitted downstream files are `10_LOCKED_NONPHOTIC_EYES_CLOSED_INTERVALS.csv` and `17_LOCKED_PREPROCESSING_ELIGIBLE_RECORDING_MANIFEST.csv`.
6. **Next after a PASS:** P1-E05 channel-name, signal-type, and montage harmonisation. Do not begin filtering, artifact correction, feature extraction, or model development before P1-E05 is locked.
7. The test split must not influence channel rules, preprocessing alternatives, representation selection, model choice, or hyperparameter tuning.
